# User-Based Collaborative Filtering (UBCF)

 User-Based CF focuses on similarity between users on a specific item. 

### Step 1: Import all necessary libraries

Import all important library before running the notebook

In [1]:
import pandas as pd
import numpy as np
import time
from math import sqrt
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity

### Step 2: Load the dataset

u.data gives the actual user-movie-rating triples, while u.item gives the movies metadata (titles, genres, etc) that is important to display human-readable recommendations instead of just providing movie IDs.

In [2]:
# Import data of users
user_df = pd.read_csv('C:/Lecture Notes/Degree/Sem 9/TDA - Algorithm and Analysis/Collaborative-Filtering-Algorithms-in-Recommendation-Systems/datasets/ml-100k/u.data', 
                        sep="\t",
                        header=None,
                        usecols=[0, 1, 2],
                        names=["user_id", "movie_id", "rating"]
                        )

# Import data of items
item_df = pd.read_csv('C:/Lecture Notes/Degree/Sem 9/TDA - Algorithm and Analysis/Collaborative-Filtering-Algorithms-in-Recommendation-Systems/datasets/ml-100k/u.item',
                        sep="|",
                        header=None,
                        encoding="latin-1",
                        names=[
                            "movie_id", "title", "release_date", "video_release_date",
                            "IMDb_URL", "unknown", "Action", "Adventure", "Animation",
                            "Children's", "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
                            "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi",
                            "Thriller", "War", "Western"
                        ])

Select 20,000 sampels out of 100,000 ratings

In [3]:
#select 20k samples
ratings_20k = user_df.head(20000)

print(f"Ratings dataset shape: {ratings_20k.shape}")

Ratings dataset shape: (20000, 3)


### Step 3: Build the user-item matrix

Pivot the long-format ratings table into a wide user-item matrix, where each row is a user, each column is a movie, and each cell contains the user's rating for the movie. This matric is the foundation for both the similarity computation and prediction step later.

In [4]:
#create the user-item matrix
user_item_matrix = ratings_20k.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)
print(user_item_matrix)

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          NaN   NaN   NaN   NaN   NaN   5.0   NaN   NaN   NaN   3.0  ...   
2          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5          4.0   3.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...        ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
457        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
458        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
459        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
460        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   3.0  ...   
462        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

Since cosine similarity does not operate on missing values, replace NaN with 0

In [5]:
#replace NaN values with 0 
user_item_matrix = user_item_matrix.fillna(0)
print(user_item_matrix.head())

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          0.0   0.0   0.0   0.0   0.0   5.0   0.0   0.0   0.0   3.0  ...   
2          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5          4.0   3.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   

movie_id  1555  1557  1561  1562  1563  1565  1578  1582  1586  1591  
user_id                                                               
1          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
2          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
5          0.0   0.0   0.0   0.0  

### Step 4: Compute the user similarity

Calculate the pairwise between every pair of users based on their rating vectors. Users with similar movies rating are close to 1, while users with little to no overlap, or have opposite ratings are closer to 0.

In [6]:
#measure similarity between users (cosine similarity)
user_similarity = cosine_similarity(user_item_matrix)
print(user_similarity)

[[1.         0.         0.01598033 ... 0.         0.06830542 0.        ]
 [0.         1.         0.08481405 ... 0.         0.         0.        ]
 [0.01598033 0.08481405 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.06830542 0.         0.         ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]]


### Step 5: Compute the user similarity matrix

Converts the raw similarity to a more readable DataFrame format

In [7]:
# Wrap in a DataFrame for better visualization
user_similarity_df = pd.DataFrame(
    user_similarity,
    index = user_item_matrix.index,
    columns = user_item_matrix.index
).round(4)

print(f"User-User Similarity Matrix shape: {user_similarity_df.shape}\n")
print(user_similarity_df.head())

User-User Similarity Matrix shape: (459, 459)

user_id     1       2       3       4      5       6       7       8    \
user_id                                                                  
1        1.0000  0.0000  0.0160  0.0165  0.191  0.2283  0.2290  0.1222   
2        0.0000  1.0000  0.0848  0.1874  0.000  0.0477  0.0240  0.1047   
3        0.0160  0.0848  1.0000  0.1574  0.000  0.0364  0.0561  0.0187   
4        0.0165  0.1874  0.1574  1.0000  0.000  0.0488  0.0402  0.1284   
5        0.1910  0.0000  0.0000  0.0000  1.000  0.1009  0.1642  0.1052   

user_id     9       10   ...     452     453     454  455     456  457  \
user_id                  ...                                             
1        0.0405  0.1245  ...  0.0283  0.0938  0.0344  0.0  0.0577  0.0   
2        0.0613  0.0172  ...  0.0000  0.0000  0.0000  0.0  0.0000  0.0   
3        0.0000  0.0105  ...  0.0000  0.0000  0.0000  0.0  0.0237  0.0   
4        0.0000  0.0270  ...  0.0000  0.0000  0.0000  0.0  0.061

### Step 6: Select the target user

Pick a user to generate the recommendations, then split their range into two groups: rated movies and unrated movies. 

In [8]:
# Change user_id to test different users
user = user_item_matrix.index[0] # Example: user_id = 1

# Get the ratings of the target user
rated_movies = user_item_matrix.loc[user]
rated_movies = rated_movies[rated_movies > 0] # Rated movies only

# Unrated movies by the target user - Potential movies for recommendation
unrated_movies = user_item_matrix.loc[user]
unrated_movies = unrated_movies[unrated_movies == 0] # Unrated movies only

print(f"Rated movies by User {user}: {len(rated_movies)}")
print(f"Unrated movies by User {user}: {len(unrated_movies)}")


Rated movies by User 1: 137
Unrated movies by User 1: 1273


### Step 7: Find the top-N similar users

For a given target user and a candidate movie, the function predict_user_ratings finds all other users who rated the movie and compare their similarity to the target user, keeps the top-N most similar, and computes a similarity-weighted average of their ratings as the predicted score. if none of the other users who rated the movie is meaningfully similar, it falls back to the target user's own average rating.

In [ ]:
def predict_user_ratings(user_id, item_id, user_item_matrix, user_similarity_df, N=10):
    #get the  users who review this specific movies
    item_ratings = user_item_matrix[item_id]

    raters = item_ratings[item_ratings>0]

    if len(raters) == 0:
        user_ratings = user_item_matrix.loc[user_id]
        return user_ratings[user_ratings>0].mean()
    
    #similarity between target user and each user who rated this movie
    user_similarity_score = user_similarity_df.loc[user_id, raters.index]

    top_n_users = user_similarity_score.sort_values(ascending=False).head(N)

    if top_n_users.sum() == 0:
        user_ratings = user_item_matrix.loc[user_id]
        return user_ratings[user_ratings>0].mean()
    
    numerator = sum(top_n_users[u] * raters[u] for u in top_n_users.index)
    denominator = top_n_users.sum()

    return numerator / denominator

### Step 8: Rank and recommend ratings based on users

Loop the prediction function over every unrated movie for the target user to generate a dictionary of predicted scores. Then, sort and format the Top-K into a readable table with along with the movie titles.

In [10]:
K = 10
N = 10

predicted_scores = {}

for item_id in unrated_movies.index:
    predicted_scores[item_id] = predict_user_ratings(
        user,
        item_id,
        user_item_matrix,
        user_similarity_df,
        N
    )

# Sort predicted scores and return top-K recommended movies
def recommend_movies(predictions, item_df, K):
    # Sort predicted scores in descending order
    top_k = dict(sorted(predictions.items(), key=lambda item: item[1], reverse=True))

    results = []
    for item_id, score in top_k.items():
        title = item_df.loc[item_df['movie_id'] == item_id, 'title'].values
        title = title[0] if len(title) > 0 else "Unknown"
        results.append({
            "movie_id": item_id,
            "title": title,
            "predicted_rating": round(score, 4)
        })
    return pd.DataFrame(results)

recommendations = recommend_movies(predicted_scores, item_df, K)
print(f"Top-{K} Recommended movies for User {user}:\n")
print(recommendations.head(K))


Top-10 Recommended movies for User 1:

   movie_id                                              title  \
0      1203                                     Top Hat (1935)   
1       119             Maya Lin: A Strong Clear Vision (1994)   
2       667                                 Audrey Rose (1977)   
3       701  Wonderful, Horrible Life of Leni Riefenstahl, ...   
4       718                      In the Bleak Midwinter (1995)   
5       814                      Great Day in Harlem, A (1994)   
6       884                           Year of the Horse (1997)   
7       909                            Dangerous Beauty (1998)   
8      1062                      Four Days in September (1997)   
9      1103                                       Trust (1990)   

   predicted_rating  
0               5.0  
1               5.0  
2               5.0  
3               5.0  
4               5.0  
5               5.0  
6               5.0  
7               5.0  
8               5.0  
9             

### Step 9: Evaluate the performance of UBCF

Split a user's ratings to 70% for train (treated as known) and 30% test (hidden from the model). The predicted ratings are compared against the true values.

In [11]:
# Test set = 30%, Train set = 70%
actual = []
predicted = []

start_time = time.time()

for user_id in user_item_matrix.index:
    # Get the ratings of the target user
    user_ratings = user_item_matrix.loc[user_id]
    rated_movies = user_ratings[user_ratings > 0] # Rated movies only

    train_set = rated_movies.sample(frac=0.7) # 70% for train set
    test_set = rated_movies.drop(train_set.index) # Remaining 30% for test set

    for movie_id in test_set.index:
        true_rating = user_item_matrix.loc[user_id, movie_id]
        user_item_matrix.loc[user_id, movie_id] = 0 # Temporarily remove the rating for prediction

        prediction = predict_user_ratings(
            user_id,
            movie_id,
            user_item_matrix,
            user_similarity_df,
            N
        )

        # Restore the original rating
        user_item_matrix.loc[user_id, movie_id] = true_rating

        actual.append(true_rating)
        predicted.append(prediction)

end_time = time.time()

print("Root Mean Squared Error (RMSE):", round(sqrt(mean_squared_error(actual, predicted)), 4))
print("Mean Absolute Error (MAE):", round(mean_absolute_error(actual, predicted), 4))
print("Execution Time:", round(end_time - start_time, 4) * 1000, "ms")

Root Mean Squared Error (RMSE): 1.0534
Mean Absolute Error (MAE): 0.8243
Execution Time: 24108.5 ms
